# RecruitGen — Model 2 of 3: CVAE (Synthetic Candidate Generation & Role Fit)

**Goal:** learn candidate representations and generate synthetic candidate profiles conditioned on a target job role.

```
Candidate Features → Preprocessing → Encoder → Latent Space → Decoder + Condition → Synthetic Candidate
```

* **Input features:** Experience, Projects, Skills, AI Score, Education, Certifications, Job Role, Recruiter Decision
* **Preprocessing:** MultiLabelBinarizer (skills) · OneHotEncoder (job role condition) · StandardScaler (numerical features)
* **Reparameterization** enables sampling from the latent space; the condition steers generation.

**Outputs:** `vae_model.h5`, `vae.weights.h5`, `scaler.json`.

## 1. Setup

In [ ]:
import os, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
import keras
from keras import layers, ops

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

QUICK_RUN = False
N_CANDIDATES = 600 if QUICK_RUN else 6000
EPOCHS = 3 if QUICK_RUN else 150
BATCH_SIZE = 64
LATENT_DIM = 8
BETA = 1.0                     # weight of the KL term

DATA_PATH = "data/candidates.csv"   # optional real dataset
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ROLES = ["AI/ML Engineer", "Cloud Engineer", "Data Analyst", "General IT", "Software Engineer"]
FEATURES = ["experience_years", "projects_count", "programming_score", "data_analytics_score", "ml_ai_score",
            "cloud_score", "communication_score", "certification_count", "education_level"]

## 2. Candidate structured dataset

Columns: Experience, Skills, Education, Certifications, Job Role, Projects, AI Score, Recruiter Decision.
If `data/candidates.csv` exists it is loaded; otherwise a realistic dataset is simulated with
role-specific skill distributions.

In [ ]:
SKILL_POOL = {
    "programming": ["Python", "Java", "C++", "JavaScript", "Go", "Git", "REST APIs", "OOP"],
    "data": ["SQL", "Excel", "Power BI", "Tableau", "Pandas", "Statistics", "Data Visualization"],
    "ml": ["TensorFlow", "PyTorch", "Scikit-learn", "NLP", "Computer Vision", "Deep Learning", "LLMs"],
    "cloud": ["AWS", "Azure", "GCP", "Docker", "Kubernetes", "Terraform", "CI/CD"],
    "soft": ["Communication", "Teamwork", "Leadership", "Presentation", "Problem Solving"],
}
ROLE_FOCUS = {
    "AI/ML Engineer":    {"programming": 4, "data": 2, "ml": 5, "cloud": 1, "soft": 2},
    "Cloud Engineer":    {"programming": 3, "data": 1, "ml": 1, "cloud": 5, "soft": 2},
    "Data Analyst":      {"programming": 2, "data": 5, "ml": 1, "cloud": 1, "soft": 3},
    "General IT":        {"programming": 2, "data": 2, "ml": 1, "cloud": 2, "soft": 3},
    "Software Engineer": {"programming": 5, "data": 1, "ml": 1, "cloud": 2, "soft": 2},
}
EDUCATION = ["Diploma", "Bachelor's", "Master's", "PhD"]

def simulate(n):
    rows = []
    for _ in range(n):
        role = random.choice(ROLES)
        skills = []
        for group, k in ROLE_FOCUS[role].items():
            skills += random.sample(SKILL_POOL[group], min(len(SKILL_POOL[group]), max(0, int(np.random.normal(k, 1)))))
        exp = round(max(0, np.random.gamma(2.0, 2.0)), 1)
        rows.append({
            "Experience (Years)": exp,
            "Skills": ", ".join(sorted(set(skills))),
            "Education": np.random.choice(EDUCATION, p=[0.1, 0.62, 0.24, 0.04]),
            "Certifications": int(np.random.poisson(1 + exp / 3)),
            "Job Role": role,
            "Projects Count": int(np.random.poisson(2 + exp / 2)),
            "AI Score (0-100)": int(np.clip(np.random.normal(60 + 6 * len(skills) / 4, 12), 0, 100)),
        })
    df = pd.DataFrame(rows)
    df["Recruiter Decision"] = np.where(df["AI Score (0-100)"] + np.random.normal(0, 8, len(df)) > 65, "Hire", "Reject")
    return df

raw = pd.read_csv(DATA_PATH) if os.path.exists(DATA_PATH) else simulate(N_CANDIDATES)
raw.head()

## 3. Preprocessing — MultiLabelBinarizer · OneHotEncoder · StandardScaler

In [ ]:
# Skills -> multi-hot matrix
mlb = MultiLabelBinarizer()
skills_mh = pd.DataFrame(mlb.fit_transform(raw["Skills"].fillna("").str.split(r",\s*")), columns=mlb.classes_)

def group_score(group):
    cols = [s for s in SKILL_POOL[group] if s in skills_mh.columns]
    return (skills_mh[cols].sum(axis=1) / len(SKILL_POOL[group]) * 100).clip(0, 100)

edu_map = {"Diploma": 1, "Bachelor's": 2, "Master's": 3, "PhD": 5}
data = pd.DataFrame({
    "experience_years": raw["Experience (Years)"].astype(float),
    "projects_count": raw["Projects Count"].astype(float),
    "programming_score": group_score("programming"),
    "data_analytics_score": group_score("data"),
    "ml_ai_score": group_score("ml"),
    "cloud_score": group_score("cloud"),
    "communication_score": group_score("soft"),
    "certification_count": raw["Certifications"].astype(float),
    "education_level": raw["Education"].map(edu_map).fillna(2).astype(float),
})

# Job role -> one-hot condition vector
ohe = OneHotEncoder(categories=[ROLES], sparse_output=False)
C = ohe.fit_transform(raw[["Job Role"]]).astype("float32")

# Numerical features -> StandardScaler
scaler = StandardScaler()
X = scaler.fit_transform(data[FEATURES]).astype("float32")

X_train, X_val, C_train, C_val = train_test_split(X, C, test_size=0.2, random_state=SEED, stratify=raw["Job Role"])
print("X", X.shape, "| condition", C.shape)
data.describe().round(2)

## 4. CVAE architecture (encoder · reparameterization · conditional decoder)

In [ ]:
@keras.saving.register_keras_serializable(package="RecruitGen")
class Sampling(layers.Layer):
    """z = mean + exp(0.5 * log_var) * epsilon"""
    def call(self, inputs):
        z_mean, z_log_var = inputs
        return z_mean + ops.exp(0.5 * z_log_var) * keras.random.normal(ops.shape(z_mean))

INPUT_DIM, CONDITION_DIM = len(FEATURES), len(ROLES)

def build_cvae():
    x_in = keras.Input((INPUT_DIM,), name="candidate_features")
    c_in = keras.Input((CONDITION_DIM,), name="target_role")
    h = layers.Concatenate(name="encoder_condition")([x_in, c_in])
    h = layers.Dense(64, activation="relu", name="encoder_dense_1")(h)
    h = layers.Dense(32, activation="relu", name="encoder_dense_2")(h)
    z_mean = layers.Dense(LATENT_DIM, name="z_mean")(h)
    z_log_var = layers.Dense(LATENT_DIM, name="z_log_var")(h)
    z = Sampling(name="latent_sampling")([z_mean, z_log_var])
    h = layers.Concatenate(name="decoder_condition")([z, c_in])
    h = layers.Dense(32, activation="relu", name="decoder_dense_1")(h)
    h = layers.Dense(64, activation="relu", name="decoder_dense_2")(h)
    out = layers.Dense(INPUT_DIM, activation="sigmoid", name="candidate_reconstruction")(h)
    return keras.Model([x_in, c_in], out, name="RecruitGen_CVAE")

cvae = build_cvae()
encoder = keras.Model(cvae.inputs, [cvae.get_layer("z_mean").output, cvae.get_layer("z_log_var").output], name="cvae_encoder")
cvae.summary()

## 5. Training — reconstruction loss + KL divergence

In [ ]:
class CVAETrainer(keras.Model):
    def __init__(self, cvae, encoder, beta=1.0):
        super().__init__()
        self.cvae, self.encoder, self.beta = cvae, encoder, beta
        self.trackers = {k: keras.metrics.Mean(name=k) for k in ["loss", "reconstruction", "kl"]}

    @property
    def metrics(self):
        return list(self.trackers.values())

    def _compute_losses(self, x, c, training):
        z_mean, z_log_var = self.encoder([x, c], training=training)
        recon = self.cvae([x, c], training=training)
        rec = ops.mean(ops.sum(ops.square(x - recon), axis=1))
        kl = ops.mean(-0.5 * ops.sum(1 + z_log_var - ops.square(z_mean) - ops.exp(z_log_var), axis=1))
        return rec + self.beta * kl, rec, kl

    def _update(self, total, rec, kl):
        for k, v in zip(["loss", "reconstruction", "kl"], [total, rec, kl]):
            self.trackers[k].update_state(v)
        return {k: m.result() for k, m in self.trackers.items()}

    def train_step(self, data):
        x, c = data[0] if len(data) == 1 else data
        with tf.GradientTape() as tape:
            total, rec, kl = self._compute_losses(x, c, True)
        grads = tape.gradient(total, self.cvae.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.cvae.trainable_weights))
        return self._update(total, rec, kl)

    def test_step(self, data):
        x, c = data[0] if len(data) == 1 else data
        return self._update(*self._compute_losses(x, c, False))

trainer = CVAETrainer(cvae, encoder, beta=BETA)
trainer.compile(optimizer=keras.optimizers.Adam(1e-3))
history = trainer.fit(
    tf.data.Dataset.from_tensor_slices((X_train, C_train)).shuffle(4096, seed=SEED).batch(BATCH_SIZE),
    validation_data=tf.data.Dataset.from_tensor_slices((X_val, C_val)).batch(BATCH_SIZE),
    epochs=EPOCHS,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)],
    verbose=2,
)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for a, k in zip(ax, ["loss", "reconstruction", "kl"]):
    a.plot(history.history[k], label="train"); a.plot(history.history["val_" + k], label="validation")
    a.set_title(k); a.set_xlabel("epoch"); a.legend(); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Evaluation — reconstruction behaviour · synthetic candidate quality · feature-distribution consistency

In [ ]:
rec_val = cvae.predict([X_val, C_val], verbose=0)
print("Validation reconstruction MSE:", float(np.mean((X_val - rec_val) ** 2)))

# Decoder only: latent z + role condition -> candidate
z_in = keras.Input((LATENT_DIM,)); c_in = keras.Input((CONDITION_DIM,))
h = cvae.get_layer("decoder_condition")([z_in, c_in])
h = cvae.get_layer("decoder_dense_2")(cvae.get_layer("decoder_dense_1")(h))
decoder = keras.Model([z_in, c_in], cvae.get_layer("candidate_reconstruction")(h), name="cvae_decoder")

def generate(role, n=500):
    c = np.zeros((n, CONDITION_DIM), "float32"); c[:, ROLES.index(role)] = 1
    z = np.random.normal(size=(n, LATENT_DIM)).astype("float32")
    return pd.DataFrame(decoder.predict([z, c], verbose=0) * scaler.scale_ + scaler.mean_, columns=FEATURES)

synthetic = {r: generate(r) for r in ROLES}
synthetic["AI/ML Engineer"].describe().round(2)

In [ ]:
fig, ax = plt.subplots(3, 3, figsize=(15, 11))
for a, f in zip(ax.flat, FEATURES):
    a.hist(data[f], bins=30, alpha=0.5, density=True, label="real")
    a.hist(pd.concat(synthetic.values())[f], bins=30, alpha=0.5, density=True, label="synthetic")
    a.set_title(f); a.legend()
plt.suptitle("Feature-distribution consistency: real vs CVAE-generated candidates"); plt.tight_layout(); plt.show()

pd.DataFrame({r: s.mean() for r, s in synthetic.items()}).round(1)

### Role-fit score  =  40% CVAE reconstruction score  +  60% role relevance

In [ ]:
def role_fit(candidate):
    x = ((np.array([candidate[f] for f in FEATURES]) - scaler.mean_) / scaler.scale_).astype("float32")[None]
    out = []
    for i, role in enumerate(ROLES):
        c = np.zeros((1, CONDITION_DIM), "float32"); c[0, i] = 1
        z_mean, _ = encoder.predict([x, c], verbose=0)
        mse = float(np.mean((x - decoder.predict([z_mean, c], verbose=0)) ** 2))
        out.append((role, round(100 / (1 + mse), 2)))
    return sorted(out, key=lambda t: -t[1])

example = {"experience_years": 1.5, "projects_count": 4, "programming_score": 30, "data_analytics_score": 60,
           "ml_ai_score": 90, "cloud_score": 45, "communication_score": 70, "certification_count": 3, "education_level": 2}
pd.DataFrame(role_fit(example), columns=["Role", "CVAE score (%)"])

## 7. Export — `vae_model.h5` · `vae.weights.h5` · `scaler.json`

In [ ]:
cvae.save(os.path.join(OUTPUT_DIR, "vae_model.h5"))
cvae.save_weights(os.path.join(OUTPUT_DIR, "vae.weights.h5"))
with open(os.path.join(OUTPUT_DIR, "scaler.json"), "w") as f:
    json.dump({"mean": scaler.mean_.tolist(), "scale": scaler.scale_.tolist()}, f, indent=2)

reloaded = keras.models.load_model(os.path.join(OUTPUT_DIR, "vae_model.h5"), compile=False, custom_objects={"Sampling": Sampling})
print("Reloaded:", reloaded.name, reloaded.input_shape, "->", reloaded.output_shape)
print("Saved:", os.listdir(OUTPUT_DIR))

Copy the three files from `outputs/` into `models/cvae/` to use them in the app.